# C2 · Conda, Git, GitHub y cuadernos reproducibles

**Curso:** Bioinformática y Biología Computacional · Universidad EAFIT  
**Duración sugerida:** 3 horas  
**Modalidad:** explicación breve → práctica guiada → reto → evidencia reproducible

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/UniversidadEAFIT/compubiol_course/blob/master/notebooks/02_reproducibilidad_git/02_conda_git_github.ipynb)

> **Continuidad del material histórico:** esta versión reemplaza y amplía `20231/Github/Intro_a_GitHub.ipynb`.

## Pregunta guía

Un análisis funciona en el computador del autor y falla en el de un colaborador. **¿Cómo separar el problema de código, datos, dependencias e historia de cambios?**

### Objetivos

- crear, activar, exportar y reconstruir un ambiente Conda;
- diferenciar Git (historia local) de GitHub (colaboración remota);
- trabajar en ramas y pull requests, sin `push` directo a la rama protegida;
- documentar entradas, parámetros, versiones y salidas de un notebook;
- recuperar cambios sin depender de copias como `final_v7_ahora_si.ipynb`.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import importlib.util

REPO_URL = "https://github.com/UniversidadEAFIT/compubiol_course.git"
COLAB_DIR = Path("/content/compubiol_course")

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ
if IN_COLAB and importlib.util.find_spec("Bio") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "biopython"], check=True)

if IN_COLAB and not COLAB_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(COLAB_DIR)], check=True)
    os.chdir(COLAB_DIR)

start = Path.cwd().resolve()
ROOT = next((p for p in [start, *start.parents] if (p / "data").is_dir() and (p / "notebooks").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError(
        "No se encontró la raíz del curso. Ejecute el notebook desde el repositorio clonado."
    )
os.chdir(ROOT)
os.environ["COURSE_ROOT"] = str(ROOT)
print(f"Raíz del curso: {ROOT}")

## 1. Ambiente reproducible

El archivo `environment.yml` declara dependencias y canales. Bioconda recomienda actualmente `conda-forge` + `bioconda` con prioridad estricta.

```bash
conda env create -f environment.yml
conda activate compubiol-2026
conda env export --from-history > environment.from-history.yml
```

`--from-history` mejora portabilidad al registrar las dependencias solicitadas, no cada biblioteca transitiva específica del sistema.

In [ ]:
import shutil, subprocess

for tool in ["git", "python", "jupyter", "blastp", "mafft"]:
    path = shutil.which(tool)
    print(f"{tool:10s}: {path or 'no disponible en este ambiente'}")

## 2. Anatomía mínima de un repositorio

- `README.md`: propósito, entradas, ejecución, salidas y limitaciones.
- `.gitignore`: artefactos, datos grandes, credenciales y resultados reconstruibles.
- `environment.yml`: dependencias.
- `LICENSE`: condiciones de reutilización.
- `CITATION.cff`: cómo citar.
- historial de commits: decisiones pequeñas y legibles.

## 3. Flujo seguro de colaboración

```bash
git clone URL
git switch -c c02-reproducibilidad-usuario
# editar
git status
git diff
git add README.md environment.yml notebook.ipynb
git commit -m "Documenta ambiente y ejecución"
git push -u origin c02-reproducibilidad-usuario
```

En GitHub, abra un pull request. La rama principal debe recibir cambios revisados y validados, no entregas personales.

> No escriba tokens en notebooks o comandos compartidos. Use el gestor de credenciales, autenticación SSH o el mecanismo institucional indicado.

## 4. Laboratorio Git totalmente local

In [ ]:
import subprocess, tempfile
from pathlib import Path

sandbox = Path(tempfile.mkdtemp(prefix="git_course_"))
def run(*args):
    result = subprocess.run(args, cwd=sandbox, text=True, capture_output=True, check=True)
    return result.stdout.strip()

run("git", "init", "-b", "main")
run("git", "config", "user.name", "Course Student")
run("git", "config", "user.email", "student@example.org")
(sandbox / "README.md").write_text("# Proyecto reproducible\n", encoding="utf-8")
run("git", "add", "README.md")
run("git", "commit", "-m", "Crea estructura inicial")
run("git", "switch", "-c", "feature/documentation")
(sandbox / "README.md").write_text("# Proyecto reproducible\n\n## Ejecución\n", encoding="utf-8")
run("git", "add", "README.md")
run("git", "commit", "-m", "Documenta ejecución")
print(run("git", "log", "--oneline", "--graph", "--all"))

In [ ]:
# Comparar la rama con main y recuperar una versión histórica sin cambiar la historia.
print(run("git", "diff", "main..feature/documentation"))
first_commit = run("git", "rev-list", "--max-parents=0", "HEAD")
print("README inicial:\n", run("git", "show", f"{first_commit}:README.md"))

### Checkpoint

Explique la diferencia entre:

- `git restore archivo`: cambia el árbol de trabajo;
- `git revert COMMIT`: crea un commit que revierte otro, apropiado para historia compartida;
- `git reset --hard`: mueve referencias y descarta cambios; no es la primera opción en una rama compartida.

## 5. Cuadernos reproducibles

GitHub muestra notebooks como documentos estáticos; las celdas interactivas deben ejecutarse en Jupyter/Colab. Antes de confirmar cambios:

1. reinicie el kernel;
2. ejecute de principio a fin;
3. revise que no haya estado oculto;
4. elimine salidas pesadas o sensibles;
5. use rutas relativas;
6. registre semillas y versiones.

In [ ]:
import platform, sys
print("Python:", sys.version.split()[0])
print("Sistema:", platform.platform())
try:
    import pandas as pd
    print("pandas:", pd.__version__)
except ImportError:
    print("pandas no instalado")

## Reto

Cree un repositorio de entrega con cuatro commits que cuenten una historia: estructura, ambiente, análisis y documentación. Abra un pull request y use la plantilla de revisión.

Referencias: documentación oficial de Conda, Bioconda y GitHub.